# SPMV CSR

In [ ]:
from pynq import Overlay

#Load the overlay
overlay = Overlay('/home/xilinx/spmv_csr_optimized/spmv_csr_optimized.bit')
print('Overlay loaded!')

In [ ]:
# Print all IPs found in the design
print(overlay.ip_dict.keys())

# Get handler of the spmv IP
spmv_ip = overlay.spmv_csr_optimized_0
print(spmv_ip)

# Print register map
print(spmv_ip.register_map)

In [ ]:
# Run the accelerator

import time
import numpy as np
from pynq import allocate

# Define Problem Dimensions
num_rows = 10
num_cols = 10
PACK_SIZE = 16  # Hardware fetches 16 elements (512-bit) at a time

# CSR Data for the 10x10 sparse matrix (19 non-zero elements)
A_values_data    = [1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 1.0]
A_col_index_data = [0, 5, 1, 7, 2, 3, 8, 1, 4, 5, 9, 0, 6, 2, 7, 8, 9, 0, 9]
A_row_index_data = [0, 2, 4, 5, 7, 9, 11, 13, 15, 17, 19]

# Input vector 'x'
x_data = [1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 10.0]

# Calculate Padding
# The hardware reads in chunks of 16. We must pad buffers to avoid out-of-bounds DDR reads.
nnz = len(A_values_data)
padded_nnz = ((nnz + PACK_SIZE - 1) // PACK_SIZE) * PACK_SIZE
padded_cols = ((num_cols + PACK_SIZE - 1) // PACK_SIZE) * PACK_SIZE

# Allocate Physical Memory (Contiguous DDR)
print(f"Allocating DDR memory: NNZ={nnz} (Padded to {padded_nnz})")

A_val_buf = allocate(shape=(padded_nnz,), dtype=np.float32)
A_col_buf = allocate(shape=(padded_nnz,), dtype=np.int32)
A_row_buf = allocate(shape=(len(A_row_index_data),), dtype=np.int32)
x_buf     = allocate(shape=(padded_cols,), dtype=np.float32)
y_buf     = allocate(shape=(num_rows,), dtype=np.float32)

# Initialize Buffers
A_val_buf[:nnz] = A_values_data
A_col_buf[:nnz] = A_col_index_data
A_row_buf[:]    = A_row_index_data
x_buf[:num_cols] = x_data

# Zero-out the padding to prevent garbage data from entering the pipeline
A_val_buf[nnz:] = 0.0
A_col_buf[nnz:] = 0
x_buf[num_cols:] = 0.0

# Flush CPU cache to DDR
A_val_buf.sync_to_device()
A_col_buf.sync_to_device()
A_row_buf.sync_to_device()
x_buf.sync_to_device()

# Setup Register Offsets (Vitis HLS 2024.2 Map)
ADDR_AP_CTRL    = 0x00
ADDR_NUM_ROWS   = 0x10
ADDR_NUM_COLS   = 0x18
ADDR_NNZ        = 0x20
ADDR_A_ROW_IDX  = 0x28
ADDR_A_COL_IDX  = 0x34
ADDR_A_VALUES   = 0x40
ADDR_X          = 0x4c
ADDR_Y          = 0x58

def write_64bit_address(ip, base_offset, address):
    """Writes a 64-bit physical address across two 32-bit registers."""
    ip.write(base_offset, address & 0xFFFFFFFF)
    ip.write(base_offset + 0x04, (address >> 32) & 0xFFFFFFFF)

# Write Scalar Parameters
spmv_ip.write(ADDR_NUM_ROWS, num_rows)
spmv_ip.write(ADDR_NUM_COLS, num_cols)
spmv_ip.write(ADDR_NNZ, nnz)

# Write 64-bit Memory Pointers
write_64bit_address(spmv_ip, ADDR_A_ROW_IDX, A_row_buf.device_address)
write_64bit_address(spmv_ip, ADDR_A_COL_IDX, A_col_buf.device_address)
write_64bit_address(spmv_ip, ADDR_A_VALUES, A_val_buf.device_address)
write_64bit_address(spmv_ip, ADDR_X, x_buf.device_address)
write_64bit_address(spmv_ip, ADDR_Y, y_buf.device_address)

# Execute Hardware
print("Starting Hardware Execution...")
start_time = time.time()

spmv_ip.write(ADDR_AP_CTRL, 0x01) # Set ap_start

# Poll ap_done (bit 1)
while (spmv_ip.read(ADDR_AP_CTRL) & 0x02) == 0:
    pass

end_time = time.time()
print(f"HW Execution Time: {(end_time - start_time) * 1000:.4f} ms")

# Retrieve and Verify Results
y_buf.sync_from_device()
y_hw = np.array(y_buf)

# Software Reference
y_sw = np.zeros(num_rows, dtype=np.float32)

sw_start = time.time()

for i in range(num_rows):
    for j in range(A_row_index_data[i], A_row_index_data[i+1]):
        y_sw[i] += A_values_data[j] * x_data[A_col_index_data[j]]

sw_end = time.time()
sw_duration = (sw_end - sw_start) * 1000
print(f"SW Execution Time: {sw_duration:.4f} ms")

print("\n--- Results ---")
print(f"Hardware Vector y: {y_hw}")
print(f"Software Vector y: {y_sw}")

if np.allclose(y_hw, y_sw, atol=1e-5):
    print("\n>>> SUCCESS: Hardware results match Software! <<<")
else:
    print("\n>>> ERROR: Mismatch detected! <<<")